In [18]:
import json
import pandas as pd
from utilities.extract_qids import extract_qids
print('p1')
from utilities.alignment import (
    align_sections,
    display_alignment,
    display_score_matrix,
    save_alignment_results,
)
print('p2')
from utilities.load_models import load_MPropositioneur_V2, load_Qwen3_8B_via_llama_cpp, load_Qwen38B_embedding
print('p3')
from utilities.omission_3 import calculate_omission
print('p4')
from utilities.propositioner_2 import extract_propositions_V2
print('p5')
from utilities.triplets import extract_triplets
print('p6')


p1
p2
p3
p4
p5
p6


In [16]:
# download the model propositioner
model_AP, tokenizer_AP = (load_MPropositioneur_V2())
model_embedding = load_Qwen38B_embedding()

model_triplets = load_Qwen3_8B_via_llama_cpp()
# Extract the two articles

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5141.84it/s]


In [3]:

data_en = extract_qids(
    "en",
    "Falkland_Islands",
    verbose=False,
) 
# data_en and data_es are lists of dictionaries
data_es = extract_qids(
    "es",
    "Islas_Malvinas_(territorio_británico_de_ultramar)",
    verbose=False,
)

with open( "data_es.json", "w", encoding="utf-8" ) as f:
    json.dump(data_es, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_es.json")

with open( "data_en.json", "w", encoding="utf-8" ) as f:
    json.dump(data_en, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_en.json")




   batch 1/11 (25 titles)
   batch 2/11 (25 titles)
   batch 3/11 (25 titles)
   batch 4/11 (25 titles)
   batch 5/11 (25 titles)
   batch 6/11 (25 titles)
   batch 7/11 (25 titles)
   batch 8/11 (25 titles)
   batch 9/11 (25 titles)
   batch 10/11 (25 titles)
   batch 11/11 (14 titles)
   batch 1/11 (25 titles)
   batch 2/11 (25 titles)
   batch 3/11 (25 titles)
   batch 4/11 (25 titles)
   batch 5/11 (25 titles)
   batch 6/11 (25 titles)
   batch 7/11 (25 titles)
   batch 8/11 (25 titles)
   batch 9/11 (25 titles)
   batch 10/11 (25 titles)
   batch 11/11 (14 titles)
   batch 1/7 (25 titles)
   batch 2/7 (25 titles)
   batch 3/7 (25 titles)
   batch 4/7 (25 titles)
   batch 5/7 (25 titles)
   batch 6/7 (25 titles)
   batch 7/7 (8 titles)
   batch 1/7 (25 titles)
   batch 2/7 (25 titles)
   batch 3/7 (25 titles)
   batch 4/7 (25 titles)
   batch 5/7 (25 titles)
   batch 6/7 (25 titles)
   batch 7/7 (8 titles)
Triples saved to data_es.json
Triples saved to data_en.json


In [4]:
# Align their sections

section_pairs, score_matrix = align_sections(
    data_en,
    data_es,
    threshold=0.70,
)

# Display results

display_alignment(section_pairs)
display_score_matrix(score_matrix)





Article A: en:Falkland_Islands
Sections A: 16
Article B: es:Islas_Malvinas_(territorio_británico_de_ultramar)
Sections B: 12

BERTScore model: bert-base-multilingual-cased
Scoring every possible section pair...
Calculating BERTScore for 99 section pairs...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6033.92it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 2/2 [00:02<00:00,  1.03s/it]


computing greedy matching.


100%|██████████| 7/7 [00:00<00:00, 74.06it/s]

done in 2.16 seconds, 45.81 sentences/sec

Finding one-to-one alignment (threshold=0.70)...

Aligned pairs: 2
Total possible pairs: 192
Unmatched A sections: 14
Unmatched B sections: 10

SECTION ALIGNMENTS

en:Q9648:1  <->  es:Q9648:1
BERTScore F1: 0.7158

en:Q9648:7  <->  es:Q9648:6
BERTScore F1: 0.7806

COMPLETE BERTSCORE MATRIX
A section                     es:Q9648:0     es:Q9648:1     es:Q9648:2     es:Q9648:3     es:Q9648:4     es:Q9648:5     es:Q9648:6     es:Q9648:7     es:Q9648:8     es:Q9648:9     es:Q9648:10    es:Q9648:11    
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
en:Q9648:0                    0.6269         0.6093         0.6428         0.6292         0.6460         0.6052         0.6598         0.6502         0.0000         0.0000         0.4876         0.0000         
en:Q9648:1                    0.64

In [5]:
propositions_list_es = []

for section in data_es["sections"]:
    propositions = extract_propositions_V2(
            section=section,
            model=model_AP,
            tokenizer=tokenizer_AP,
            chunk_max_tokens=200,
            max_new_tokens=2048)


        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_es.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_es.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_es, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_es)}")
print(f"Saved to: propositions_es.json")
print("-----------------------------------")



Processing section: es:Q9648:0
Number of chunks: 7

----------------------------------------
Processing chunk 1/7
Chunk tokens: 96
Input tokens: 107
Generated tokens: 164
Propositions: 6
Final propositions for chunk: 6

----------------------------------------
Processing chunk 2/7
Chunk tokens: 26
Input tokens: 37
Generated tokens: 41
Propositions: 3
Final propositions for chunk: 3

----------------------------------------
Processing chunk 3/7
Chunk tokens: 34
Input tokens: 45
Generated tokens: 51
Propositions: 2
Final propositions for chunk: 2

----------------------------------------
Processing chunk 4/7
Chunk tokens: 59
Input tokens: 70
Generated tokens: 111
Propositions: 4
Final propositions for chunk: 4

----------------------------------------
Processing chunk 5/7
Chunk tokens: 43
Input tokens: 54
Generated tokens: 63
Propositions: 2
Final propositions for chunk: 2

----------------------------------------
Processing chunk 6/7
Chunk tokens: 118
Input tokens: 129
Generated tokens

In [6]:

propositions_list_en = []

for section in data_en["sections"]:
    propositions = extract_propositions_V2(
            section=section,
            model=model_AP,
            tokenizer=tokenizer_AP,
            chunk_max_tokens=200,
            max_new_tokens=2048)

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_en.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_en.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_en, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_en)}")
print(f"Saved to: propositions_en.json")
print("-----------------------------------")




Processing section: en:Q9648:0
Number of chunks: 20

----------------------------------------
Processing chunk 1/20
Chunk tokens: 74
Input tokens: 85
Generated tokens: 82
Propositions: 5
Final propositions for chunk: 5

----------------------------------------
Processing chunk 2/20
Chunk tokens: 69
Input tokens: 80
Generated tokens: 113
Propositions: 6
Final propositions for chunk: 6

----------------------------------------
Processing chunk 3/20
Chunk tokens: 48
Input tokens: 59
Generated tokens: 71
Propositions: 5
Final propositions for chunk: 5

----------------------------------------
Processing chunk 4/20
Chunk tokens: 30
Input tokens: 41
Generated tokens: 50
Propositions: 4
Final propositions for chunk: 4

----------------------------------------
Processing chunk 5/20
Chunk tokens: 12
Input tokens: 23
Generated tokens: 22
Propositions: 2
Final propositions for chunk: 2

----------------------------------------
Processing chunk 6/20
Chunk tokens: 22
Input tokens: 33
Generated tok

In [13]:
propositions_list_en[0]

{'proposition_id': 'en:Q9648:0:ap:0',
 'section_id': 'en:Q9648:0',
 'article_id': 'en:Q9648',
 'language': 'en',
 'text': 'The Falkland Islands are commonly referred to as the Falklands.',
 'source_text': 'The Falkland Islands ( / ˈ f ɔː ( l ) k l ə n d , ˈ f ɒ l k -/ ; Spanish: Islas Malvinas [ˈislas malˈβinas] ), commonly referred to as the Falklands , is an archipelago in the South Atlantic Ocean on the Patagonian Shelf .'}

In [12]:
propositions_list_es[0]

{'proposition_id': 'es:Q9648:0:ap:0',
 'section_id': 'es:Q9648:0',
 'article_id': 'es:Q9648',
 'language': 'es',
 'text': 'El territorio británico de ultramar de las Islas Malvinas es un territorio dependiente.',
 'source_text': 'El territorio británico de ultramar de las Islas Malvinas \u200b (en inglés : British Overseas Territory of the Falkland Islands ) es un territorio dependiente y no autónomo , bajo administración del Reino Unido , que abarca la totalidad del archipiélago de las Malvinas , situado en el océano Atlántico sudoccidental , en el extremo sudeste de América del Sur .'}

In [19]:
# manual
list_manual_section_pairs = [
    {
        "section_id_a": "es:Q9648:0",
        "section_id_b": "en:Q9648:0",
        "pair_id": "es:Q9648:0+en:Q9648:0+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:1",
        "section_id_b": "en:Q9648:1",
        "pair_id": "es:Q9648:1+en:Q9648:1+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:2",
        "section_id_b": "en:Q9648:2",
        "pair_id": "es:Q9648:2+en:Q9648:2+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:3",
        "section_id_b": "en:Q9648:3",
        "pair_id": "es:Q9648:3+en:Q9648:3+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:5",
        "section_id_b": "en:Q9648:6",
        "pair_id": "es:Q9648:5+en:Q9648:6+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:6",
        "section_id_b": "en:Q9648:7",
        "pair_id": "es:Q9648:6+en:Q9648:7+Manual",
        "alignment_score": "Manual"
    }
]

In [26]:
results_omission = []

for pair in list_manual_section_pairs:
    section_pair_id = pair["pair_id"]
    section_id_en = pair["section_id_b"]
    section_id_es = pair["section_id_a"]

    section_aps_en = [
        {
            "proposition_id": ap["proposition_id"],
            "text": ap["text"]
        }
        for ap in propositions_list_en
        if section_id_en in ap["proposition_id"]
    ]

    section_aps_es = [
        {
            "proposition_id": ap["proposition_id"],
            "text": ap["text"]
        }
        for ap in propositions_list_es
        if section_id_es in ap["proposition_id"]
    ]

    print(
        f"{section_id_en} (EN) vs {section_id_es} (ES) | "
        f"EN: {len(section_aps_en)} | ES: {len(section_aps_es)}"
    )

    omission_metrics = calculate_omission(
        section_aps_en,
        section_aps_es,
        threshold=0.73,
        model=model_embedding
    )

    results_omission.append({
        "section_pair_id": section_pair_id,
        "section_en": { "section_id": section_id_en, "propositions": section_aps_en },
        "section_es": { "section_id": section_id_es, "propositions": section_aps_es },
        "omission_metrics": omission_metrics})

with open("omission_results.json", "w", encoding="utf-8") as file:
    json.dump(results_omission, file, ensure_ascii=False, indent=4)


en:Q9648:0 (EN) vs es:Q9648:0 (ES) | EN: 60 | ES: 25
en:Q9648:1 (EN) vs es:Q9648:1 (ES) | EN: 45 | ES: 114
en:Q9648:2 (EN) vs es:Q9648:2 (ES) | EN: 196 | ES: 136
en:Q9648:3 (EN) vs es:Q9648:3 (ES) | EN: 88 | ES: 64
en:Q9648:6 (EN) vs es:Q9648:5 (ES) | EN: 86 | ES: 27
en:Q9648:7 (EN) vs es:Q9648:6 (ES) | EN: 64 | ES: 71


In [27]:
with open("omission_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)


equivalent_aps = {}


for section in results_omission:
    section_pair_id = section["section_pair_id"]
    metrics = section["omission_metrics"]

    # ---------------------------------------------------------
    # EN -> ES
    # ---------------------------------------------------------

    for match_group in metrics["en_to_es"]["matches"]:
        ap_en = match_group["ap"]

        for match in match_group["matches"]:
            ap_es = match["ap"]

            key = (
                section_pair_id,
                ap_en["proposition_id"],
                ap_es["proposition_id"]
            )

            equivalent_aps[key] = {
                "section_pair_id": section_pair_id,
                "ap_en_id": ap_en["proposition_id"],
                "ap_en_text": ap_en["text"],
                "ap_es_id": ap_es["proposition_id"],
                "ap_es_text": ap_es["text"],
                "similarity": match["similarity"]
            }


    # ---------------------------------------------------------
    # ES -> EN
    # ---------------------------------------------------------

    for match_group in metrics["es_to_en"]["matches"]:
        ap_es = match_group["ap"]

        for match in match_group["matches"]:
            ap_en = match["ap"]

            key = (
                section_pair_id,
                ap_en["proposition_id"],
                ap_es["proposition_id"]
            )

            # Solo añadir si no estaba ya en EN -> ES
            if key not in equivalent_aps:
                equivalent_aps[key] = {
                    "section_pair_id": section_pair_id,
                    "ap_en_id": ap_en["proposition_id"],
                    "ap_en_text": ap_en["text"],
                    "ap_es_id": ap_es["proposition_id"],
                    "ap_es_text": ap_es["text"],
                    "similarity": match["similarity"]
                }


# Convertir a lista
equivalent_aps = list(equivalent_aps.values())

with open("equivalent_aps.json", "w", encoding="utf-8") as f:
    json.dump(equivalent_aps, f, ensure_ascii=False, indent=4)